In [ ]:
#!/usr/bin/env python3
"""
RSA Lag Analysis — All Regions, Band × Pair (Long Format)
----------------------------------------------------------
Regions : LHP, RHP, LLTC, RLTC, LMTL, RMTL

Bands   : theta  (4–8 Hz,    freq idx 2–4  → 4.67, 5.82, 7.26 Hz)
          alpha  (8–12 Hz,   freq idx 4–6  → 7.26, 9.05, 11.28 Hz)
          beta   (12–40 Hz,  freq idx 7–11 → 14.07–34.03 Hz)
          gamma  (40–128 Hz, freq idx 12–17→ 42.44–128.00 Hz)

  NOTE on theta/alpha boundary: 7.26 Hz (idx 4) falls in both bands.
  This is intentional — theta ends at ~8 Hz, alpha starts at ~8 Hz,
  and the nearest log-spaced bin straddles the boundary.
  If you want strict non-overlapping bands, remove idx 4 from one.

For every trial × region × recall pair (i, j) with i < j × band:

  RSA_r_ret         : Pearson r between retrieval osc vectors at output pos i and j
  RSA_r_enc         : Pearson r between encoding  osc vectors at serial pos of i and j
  RSA_r_enc_i_ret_j : Pearson r between encoding  osc of word i  ×  retrieval osc of word j
  RSA_r_enc_j_ret_i : Pearson r between encoding  osc of word j  ×  retrieval osc of word i

Output (one master CSV per experiment):
  ./rsa_lag_allregions/ALL_SUBJECTS_{exp}_allregions_allbands_rsa_lag.csv

18 log-spaced frequencies (3–128 Hz):
  idx:  0     1     2     3     4     5      6      7      8
  Hz:  3.00  3.74  4.67  5.82  7.26  9.05  11.28  14.07  17.55
  idx:  9     10    11    12    13    14    15     16     17
  Hz: 21.88 27.29 34.03 42.44 52.92 66.00 82.31 102.64 128.00

FR1 notes:
  - No spatial task → SP_lag is always NaN
  - No stimulation columns
  - Input: ./subject_results_FR1/ALL_SUBJECTS_FR1_irasa_channel_wide.csv
  - Upstream pipeline already filters to clean recalls
    (recalled=True, intrusion=0, deduplicated by item_num)
"""

import warnings
import traceback
from pathlib import Path
from typing import List, Dict, Optional

import numpy as np
import pandas as pd
from scipy.stats import pearsonr
from scipy.spatial.distance import euclidean

warnings.filterwarnings('ignore')

# ============================================================================
# CONFIGURATION
# ============================================================================

EXPERIMENTS = ['FR1']

INPUT_DIRS = {
    
    'FR1':                 Path('./subject_results_FR1'),
}

# Per-experiment master CSV filename stem.
# For FR1 the upstream script uses a different naming convention.
INPUT_FILENAMES = {
    'DBOY1':               'ALL_SUBJECTS_DBOY1_irasa_channel_wide.csv',
    'EFRCourierReadOnly':  'ALL_SUBJECTS_EFRCourierReadOnly_irasa_channel_wide.csv',
    'EFRCourierOpenLoop':  'ALL_SUBJECTS_EFRCourierOpenLoop_irasa_channel_wide.csv',
    'FR1':                 'ALL_SUBJECTS_FR1_irasa_channel_wide.csv',
}

# Experiments that have a spatial task and therefore store_x / store_z columns.
# FR1 is purely verbal — SP_lag will be NaN throughout.
SPATIAL_EXPERIMENTS = {'DBOY1', 'EFRCourierReadOnly', 'EFRCourierOpenLoop'}

OUTPUT_DIR = Path('./rsa_lag_allregions')
OUTPUT_DIR.mkdir(exist_ok=True)

# All six regions
REGIONS = ['LHP', 'RHP', 'LLTC', 'RLTC', 'LMTL', 'RMTL']

HEMISPHERE = {
    'LHP':  'left',
    'RHP':  'right',
    'LLTC': 'left',
    'RLTC': 'right',
    'LMTL': 'left',
    'RMTL': 'right',
}

# 18 log-spaced frequencies 3–128 Hz
N_FREQS = 18

# Band definitions — freq indices into the 18-element IRASA vector
# theta: 4–8 Hz  → indices 2,3,4  (4.67, 5.82, 7.26 Hz)
# alpha: 8–12 Hz → indices 4,5,6  (7.26, 9.05, 11.28 Hz)
# beta : 12–40 Hz→ indices 7–11   (14.07–34.03 Hz)
# gamma: 40+     → indices 12–17  (42.44–128.00 Hz)
BANDS = {
    'theta': list(range(2, 5)),   # 4.67 – 7.26 Hz
    'alpha': list(range(4, 7)),   # 7.26 – 11.28 Hz
    'beta':  list(range(7, 12)),  # 14.07 – 34.03 Hz
    'gamma': list(range(12, 18)), # 42.44 – 128.00 Hz
}
BAND_ORDER = ['theta', 'alpha', 'beta', 'gamma']

RET_OSC_COLS = [f'ret_osc_f{i:02d}' for i in range(N_FREQS)]
ENC_OSC_COLS = [f'enc_osc_f{i:02d}' for i in range(N_FREQS)]

# Word2Vec
WORD2VEC_PATH = Path('/home1/noaherz/word2vec/GoogleNews-vectors-negative300.bin.gz')

OUTPUT_COLS = [
    'subject', 'session', 'experiment', 'trial',
    'region', 'hemisphere',
    'band', 'band_freq_indices',
    'output_pos_i', 'output_pos_j', 'output_lag',
    'word_i', 'word_j',
    'serial_pos_i', 'serial_pos_j', 'T_lag', 'SP_lag',
    'RSA_r_ret',          # ret_i  × ret_j
    'RSA_r_enc',          # enc_i  × enc_j
    'RSA_r_enc_i_ret_j',  # enc_i  × ret_j
    'RSA_r_enc_j_ret_i',  # enc_j  × ret_i
    'n_channels', 'semantic_sim',
]

# ============================================================================
# WORD2VEC
# ============================================================================

def load_word2vec(path: Path):
    if path is None or not path.exists():
        print(f"  ⚠  Word2Vec model not found at {path}. semantic_sim will be NaN.")
        return None
    try:
        import gensim.models as gensim_models
        print(f"  Loading Word2Vec from {path} …")
        model = gensim_models.KeyedVectors.load_word2vec_format(str(path), binary=True)
        try:
            vs = len(model)
        except TypeError:
            vs = len(model.vocab)
        print(f"  ✓ Word2Vec loaded — vocab: {vs:,}")
        return model
    except Exception as e:
        print(f"  ✗ Word2Vec load failed: {e}. semantic_sim will be NaN.")
        return None


def case_insensitive_similarity(word1: str, word2: str, model) -> Optional[float]:
    cases = [
        (word1.lower(), word2.lower()),
        (word1.lower(), word2.upper()),
        (word1.upper(), word2.lower()),
        (word1.upper(), word2.upper()),
    ]
    sims = []
    for w1, w2 in cases:
        try:
            sims.append(model.similarity(w1, w2))
        except KeyError:
            continue
    return float(max(sims)) if sims else None


def build_similarity_cache(words: set, model) -> dict:
    if model is None:
        return {}
    unique_words = sorted(w for w in words if isinstance(w, str))
    n = len(unique_words)
    print(f"    Building semsim cache: {n} words → {n*(n-1)//2} pairs …")
    cache = {}
    for i, w1 in enumerate(unique_words):
        for w2 in unique_words[i:]:
            key = frozenset({w1, w2})
            if key not in cache:
                sim = case_insensitive_similarity(w1, w2, model)
                cache[key] = sim if sim is not None else np.nan
    return cache


# ============================================================================
# VECTOR + STATISTICS HELPERS
# ============================================================================

def build_band_vector(recall_rows: pd.DataFrame,
                      channel_index: list,
                      band_freq_indices: list,
                      phase_cols: list) -> np.ndarray:
    """
    Build a 1-D vector for one event and one band:
      1. Align rows to channel_index       → (N_ch, 18) matrix
      2. Slice to band_freq_indices        → (N_ch, N_band_freqs) sub-matrix
      3. Flatten                           → (N_ch * N_band_freqs,) vector

    phase_cols : either RET_OSC_COLS or ENC_OSC_COLS (18 elements)
    Missing channels filled with NaN, handled pairwise in safe_pearsonr.
    """
    ch_df = (
        recall_rows
        .drop_duplicates(subset='channel_idx')
        .set_index('channel_idx')
        .reindex(channel_index)
    )
    band_cols = [phase_cols[i] for i in band_freq_indices]
    mat = ch_df[band_cols].values   # (N_ch, N_band_freqs)
    return mat.flatten()


def safe_pearsonr(v1: np.ndarray, v2: np.ndarray) -> float:
    """Pearson r ignoring NaN positions; returns NaN if fewer than 3 valid pairs."""
    if len(v1) != len(v2):
        return np.nan
    mask = np.isfinite(v1) & np.isfinite(v2)
    if mask.sum() < 3:
        return np.nan
    r, _ = pearsonr(v1[mask], v2[mask])
    return float(r)


def safe_euclidean(x1, z1, x2, z2) -> float:
    if any(not np.isfinite(v) for v in (x1, z1, x2, z2)):
        return np.nan
    return float(euclidean([x1, z1], [x2, z2]))


def extract_scalar(series: pd.Series, field: str, context: str):
    unique_vals = series.dropna().unique()
    if len(unique_vals) > 1:
        warnings.warn(
            f"[{context}] '{field}' has {len(unique_vals)} distinct values "
            f"({unique_vals[:3]}…). Taking first.")
    return series.iloc[0]


# ============================================================================
# TRIAL-LEVEL PROCESSING
# ============================================================================

def process_trial_region(trial_df: pd.DataFrame,
                         region: str,
                         sim_cache: dict,
                         has_spatial: bool) -> List[Dict]:
    """
    For one (subject, session, trial, region):
      - Enumerate all valid recall pairs (i, j), i < j (by output position)
      - For each pair × each band compute 4 RSA measures:
            RSA_r_ret         : ret_i  × ret_j
            RSA_r_enc         : enc_i  × enc_j
            RSA_r_enc_i_ret_j : enc_i  × ret_j
            RSA_r_enc_j_ret_i : enc_j  × ret_i
      - Returns list of dicts in OUTPUT_COLS order (long format)

    has_spatial : if False (e.g. FR1), store_x/store_z are absent and
                  SP_lag is set to NaN without attempting column access.

    Encoding vectors carry the IRASA values from the encoding epoch of the
    presented word at that serial position, as written by the channel-wide
    pipeline — so enc_osc on the rows for output_pos == op_i already corresponds
    to the encoding of that recalled word.
    """
    rows = []

    output_positions = sorted(
        trial_df['recall_output_position'].unique(),
        key=lambda x: int(x),
    )
    if len(output_positions) < 2:
        return rows

    channel_index = sorted(trial_df['channel_idx'].unique(), key=int)
    sample_row    = trial_df.iloc[0]

    # ------------------------------------------------------------------
    # Pre-compute per-output-position metadata + per-band vectors
    # (retrieval AND encoding) so we only build each vector once
    # ------------------------------------------------------------------
    pos_data: Dict[int, Dict] = {}

    for op in output_positions:
        op_rows = trial_df[trial_df['recall_output_position'] == op]
        ctx = (f"subj={sample_row['subject']} sess={sample_row['session']} "
               f"trial={sample_row['trial']} region={region} op={op}")

        word       = extract_scalar(op_rows['recalled_word'],   'recalled_word',   ctx)
        serial_pos = extract_scalar(op_rows['serial_position'], 'serial_position', ctx)
        n_ch       = op_rows['channel_idx'].nunique()

        # Spatial coordinates — only accessed when the experiment has a spatial task
        if has_spatial:
            store_x = extract_scalar(op_rows['store_x'], 'store_x', ctx)
            store_z = extract_scalar(op_rows['store_z'], 'store_z', ctx)
        else:
            store_x = np.nan
            store_z = np.nan

        ret_band_vectors = {
            band_name: build_band_vector(op_rows, channel_index, freq_idx, RET_OSC_COLS)
            for band_name, freq_idx in BANDS.items()
        }
        enc_band_vectors = {
            band_name: build_band_vector(op_rows, channel_index, freq_idx, ENC_OSC_COLS)
            for band_name, freq_idx in BANDS.items()
        }

        pos_data[op] = {
            'ret_band_vectors': ret_band_vectors,
            'enc_band_vectors': enc_band_vectors,
            'word':             word,
            'serial_pos':       serial_pos,
            'store_x':          store_x,
            'store_z':          store_z,
            'n_channels':       n_ch,
        }

    subject    = sample_row['subject']
    session    = sample_row['session']
    experiment = sample_row['experiment']   # injected upstream before groupby
    trial      = sample_row['trial']
    hemi       = HEMISPHERE.get(region, 'unknown')

    # ------------------------------------------------------------------
    # Enumerate pairs × bands
    # ------------------------------------------------------------------
    for idx_i, op_i in enumerate(output_positions):
        for op_j in output_positions[idx_i + 1:]:
            d_i = pos_data[op_i]
            d_j = pos_data[op_j]

            output_lag = int(op_j) - int(op_i)
            T_lag      = abs(int(d_i['serial_pos']) - int(d_j['serial_pos']))
            SP_lag     = safe_euclidean(d_i['store_x'], d_i['store_z'],
                                        d_j['store_x'], d_j['store_z'])
            n_channels = min(d_i['n_channels'], d_j['n_channels'])

            # Semantic similarity — identical across bands, compute once per pair
            w_i = str(d_i['word']).lower() if pd.notna(d_i['word']) else None
            w_j = str(d_j['word']).lower() if pd.notna(d_j['word']) else None
            sem_sim = (
                sim_cache.get(frozenset({w_i, w_j}), np.nan)
                if (w_i and w_j and sim_cache)
                else np.nan
            )

            # One row per band
            for band_name in BAND_ORDER:
                freq_idx = BANDS[band_name]

                v_ret_i = d_i['ret_band_vectors'][band_name]
                v_ret_j = d_j['ret_band_vectors'][band_name]
                v_enc_i = d_i['enc_band_vectors'][band_name]
                v_enc_j = d_j['enc_band_vectors'][band_name]

                rows.append({
                    'subject':           subject,
                    'session':           session,
                    'experiment':        experiment,
                    'trial':             trial,
                    'region':            region,
                    'hemisphere':        hemi,
                    'band':              band_name,
                    'band_freq_indices': str(freq_idx),
                    'output_pos_i':      op_i,
                    'output_pos_j':      op_j,
                    'output_lag':        output_lag,
                    'word_i':            d_i['word'],
                    'word_j':            d_j['word'],
                    'serial_pos_i':      d_i['serial_pos'],
                    'serial_pos_j':      d_j['serial_pos'],
                    'T_lag':             T_lag,
                    'SP_lag':            SP_lag,
                    'RSA_r_ret':          safe_pearsonr(v_ret_i, v_ret_j),
                    'RSA_r_enc':          safe_pearsonr(v_enc_i, v_enc_j),
                    'RSA_r_enc_i_ret_j':  safe_pearsonr(v_enc_i, v_ret_j),
                    'RSA_r_enc_j_ret_i':  safe_pearsonr(v_enc_j, v_ret_i),
                    'n_channels':        n_channels,
                    'semantic_sim':      sem_sim,
                })

    return rows


# ============================================================================
# PER-EXPERIMENT RUNNER
# ============================================================================

def run_experiment(exp: str, w2v_model):
    print(f"\n{'='*70}")
    print(f"EXPERIMENT: {exp}")
    print(f"{'='*70}")

    input_dir  = INPUT_DIRS.get(exp)
    if input_dir is None:
        print(f"  ✗ No INPUT_DIR configured for '{exp}'.")
        return

    filename   = INPUT_FILENAMES.get(exp, f"ALL_SUBJECTS_{exp}_irasa_channel_wide.csv")
    input_path = input_dir / filename
    if not input_path.exists():
        print(f"  ✗ Master CSV not found: {input_path}")
        return

    has_spatial = exp in SPATIAL_EXPERIMENTS

    print(f"  Loading {input_path} …")
    df = pd.read_csv(input_path)
    print(f"  Loaded {len(df):,} rows | "
          f"{df['subject'].nunique()} subjects | "
          f"{df['session'].nunique()} sessions")

    # ── Inject 'experiment' column if absent (FR1 upstream doesn't write it) ──
    if 'experiment' not in df.columns:
        df['experiment'] = exp
        print(f"  ℹ  'experiment' column not found — injected as '{exp}'")

    # Filter to the 6 regions upfront
    df = df[df['region'].isin(REGIONS)].copy()
    print(f"  After region filter ({REGIONS}): {len(df):,} rows")
    if df.empty:
        print("  ✗ No matching data — check 'region' column values.")
        print(f"    Regions present in file: {df['region'].unique().tolist()}")
        return

    df['channel_idx'] = df['channel_idx'].astype(int)

    if not has_spatial:
        print(f"  ℹ  No spatial task for {exp} — SP_lag will be NaN throughout")

    # Build Word2Vec cache once per experiment across all regions
    all_words_lower = set(
        df['recalled_word'].dropna().astype(str).str.lower().unique()
    )
    sim_cache = build_similarity_cache(all_words_lower, w2v_model)

    all_region_rows = []

    for region in REGIONS:
        print(f"\n  {'─'*60}")
        print(f"  Region: {region}")

        region_df = df[df['region'] == region].copy()
        if region_df.empty:
            print(f"  ✗ No rows for region {region} — skipping")
            continue

        print(f"  Rows: {len(region_df):,}")

        all_rows = []
        groups   = region_df.groupby(['subject', 'session', 'trial'])
        n_groups = len(groups)

        for g_idx, ((subj, sess, trial), trial_df) in enumerate(groups):
            if g_idx % 200 == 0:
                print(f"    Processing trial group {g_idx}/{n_groups} …")
            try:
                rows = process_trial_region(trial_df, region, sim_cache, has_spatial)
                all_rows.extend(rows)
            except Exception as e:
                print(f"    FAILED [{subj} sess={sess} trial={trial}]: {e}")
                traceback.print_exc()
                continue

        if not all_rows:
            print(f"  ✗ No pairs generated for region {region}")
            continue

        result_df = pd.DataFrame(all_rows, columns=OUTPUT_COLS)
        all_region_rows.append(result_df)

        # Per-region summary
        print(f"  ✓ {region}: {len(result_df):,} pair-band rows")
        for band_name in BAND_ORDER:
            b = result_df[result_df['band'] == band_name]
            print(
                f"    {band_name:6s}: {len(b):7,} pairs | "
                f"RSA_ret NaN={b['RSA_r_ret'].isna().mean()*100:.1f}% | "
                f"RSA_enc NaN={b['RSA_r_enc'].isna().mean()*100:.1f}% | "
                f"cross NaN={b['RSA_r_enc_i_ret_j'].isna().mean()*100:.1f}%"
            )

    # ── Master CSV: all regions × all bands ──────────────────────────────────
    if all_region_rows:
        master_df  = pd.concat(all_region_rows, ignore_index=True)
        master_path = OUTPUT_DIR / f"ALL_SUBJECTS_{exp}_allregions_allbands_rsa_lag.csv"
        master_df.to_csv(master_path, index=False)

        print(f"\n  {'='*60}")
        print(f"  ✓ Master CSV → {master_path.name}")
        print(f"    Rows       : {len(master_df):,}")
        print(f"    Subjects   : {master_df['subject'].nunique()}")
        print(f"    Sessions   : {master_df['session'].nunique()}")
        print(f"    Regions    : {sorted(master_df['region'].unique())}")
        print(f"    Bands      : {master_df['band'].unique().tolist()}")
        print(f"    RSA_ret  NaN: {master_df['RSA_r_ret'].isna().mean()*100:.1f}%")
        print(f"    RSA_enc  NaN: {master_df['RSA_r_enc'].isna().mean()*100:.1f}%")
        print(f"    cross_ij NaN: {master_df['RSA_r_enc_i_ret_j'].isna().mean()*100:.1f}%")
        print(f"    cross_ji NaN: {master_df['RSA_r_enc_j_ret_i'].isna().mean()*100:.1f}%")
        print(f"    SemSim   NaN: {master_df['semantic_sim'].isna().mean()*100:.1f}%")
        if not has_spatial:
            print(f"    SP_lag     : all NaN (no spatial task)")
    else:
        print(f"\n  ✗ No rows produced for {exp} — master CSV not written")


# ============================================================================
# MAIN
# ============================================================================

if __name__ == '__main__':
    w2v_model = load_word2vec(WORD2VEC_PATH)

    for exp in EXPERIMENTS:
        run_experiment(exp, w2v_model)

    print(f"\n{'='*70}")
    print("✓ ALL EXPERIMENTS COMPLETE")
    print(f"{'='*70}")

In [ ]:
#!/usr/bin/env python3
"""
Merge individual session CSVs into ALL_SUBJECTS_FR1_irasa_channel_wide.csv
--------------------------------------------------------------------------
Scans ./subject_results_FR1/ for all *_FR1_irasa_channel_wide.csv files,
concatenates them, and writes the master CSV.

Safe to re-run at any time — always rebuilds from whatever session files
are currently present, so it works whether the upstream pipeline is still
running (partial merge) or fully complete.
"""

from pathlib import Path
import pandas as pd

OUTPUT_DIR  = Path('./subject_results_FR1')
MASTER_NAME = 'ALL_SUBJECTS_FR1_irasa_channel_wide.csv'
MASTER_PATH = OUTPUT_DIR / MASTER_NAME

# ── Collect session files (exclude any pre-existing master) ──────────────────
session_files = sorted(
    f for f in OUTPUT_DIR.glob('*_FR1_irasa_channel_wide.csv')
    if f.name != MASTER_NAME
)

print(f"Found {len(session_files)} session CSV(s) in {OUTPUT_DIR}")

if not session_files:
    print("Nothing to merge — exiting.")
else:
    dfs = []
    for fpath in session_files:
        try:
            df = pd.read_csv(fpath)
            dfs.append(df)
            print(f"  ✓ {fpath.name:60s}  {len(df):>7,} rows")
        except Exception as e:
            print(f"  ✗ {fpath.name}: {e}")

    if dfs:
        master = pd.concat(dfs, ignore_index=True)
        master.to_csv(MASTER_PATH, index=False)

        print(f"\n{'='*60}")
        print(f"Master CSV → {MASTER_PATH}")
        print(f"  Subjects  : {master['subject'].nunique()}")
        print(f"  Sessions  : {master['session'].nunique()}")
        print(f"  Channels  : {master['channel_label'].nunique()}")
        print(f"  Regions   : {sorted(master['region'].unique())}")
        print(f"  Rows      : {len(master):,}")
        print(f"{'='*60}")
    else:
        print("No CSVs could be read — master not written.")

In [ ]:
#!/usr/bin/env python3
"""
LMM Analysis: T_lag / SP_lag → RSA (all measures)
All Regions × All Bands × All Outcomes
----------------------------------------------------------------------
Input : ./rsa_lag_allregions/ALL_SUBJECTS_{exp}_allregions_allbands_rsa_lag.csv

Regions  : LHP, RHP, LLTC, RLTC, LMTL, RMTL
Bands    : theta, alpha, beta, gamma
Outcomes : RSA_r_ret         — retrieval  × retrieval  (both words at retrieval)
           RSA_r_enc         — encoding   × encoding   (both words at encoding)
           RSA_r_enc_i_ret_j — encoding of word i × retrieval of word j
           RSA_r_enc_j_ret_i — encoding of word j × retrieval of word i

Models (per outcome × predictor × band × region):
  Model 1 (bare)      : outcome ~ predictor
  Model 2 (controlled): outcome ~ predictor + cross_covariate + output_lag [+ semantic_sim]

  For FR1 (no spatial task): SP_lag is absent — only T_lag is used as predictor.
  Model 2 for T_lag then controls for output_lag [+ semantic_sim] only (no SP_lag
  cross-covariate), because there is no meaningful SP_lag to partial out.

Random effects: nested session within subject
  groups     = subject
  vc_formula = {'subj_sess': '0 + C(subj_sess)'}

FDR: BH correction within each model across its terms.

Outputs (per experiment, under ./rsa_lag_lmm_allregions/{experiment}/):
  LMM_{region}_{predictor}_{band}_{outcome}_results.csv / .txt
  plots/forest_{...}.png
  plots/interaction_{...}.png
  plots/summary_heatmap_{region}_{outcome}.png
  plots/beta_bars_{region}_{outcome}.png
  LMM_ALL_results.csv   — master file, all regions × outcomes × bands

Usage
-----
Run all experiments defined in EXPERIMENTS_TO_RUN, or pass one on the command line:
    python rsa_lag_lmm_allregions_fr1.py FR1
    python rsa_lag_lmm_allregions_fr1.py DBOY1
"""

import sys
import warnings
from pathlib import Path
from typing import List, Dict, Tuple, Optional

import numpy as np
import pandas as pd
import matplotlib

import matplotlib.pyplot as plt
from scipy.stats import pearsonr, ttest_1samp
from statsmodels.regression.mixed_linear_model import MixedLM
from statsmodels.stats.multitest import fdrcorrection

warnings.filterwarnings('ignore')

# ============================================================================
# EXPERIMENT REGISTRY
# ============================================================================

# All experiments this script knows about.
# Add new ones here; no other changes needed.
EXPERIMENT_CONFIG = {
    'DBOY1': {
        'has_spatial': True,
    },
    'EFRCourierReadOnly': {
        'has_spatial': True,
    },
    'EFRCourierOpenLoop': {
        'has_spatial': True,
    },
    'FR1': {
        'has_spatial': False,   # verbal free recall — no store_x / store_z
    },
}

# Which experiments to run when no command-line argument is given.
EXPERIMENTS_TO_RUN = ['FR1']

INPUT_DIR  = Path('./rsa_lag_allregions')
BASE_OUTPUT_DIR = Path('./rsa_lag_lmm_allregions')

# ============================================================================
# ANALYSIS CONSTANTS
# ============================================================================

REGIONS = ['LHP', 'RHP', 'LLTC', 'RLTC', 'LMTL', 'RMTL']

OUTCOMES = [
    'RSA_r_ret',
    'RSA_r_enc',
    'RSA_r_enc_i_ret_j',
    'RSA_r_enc_j_ret_i',
]
OUTCOME_LABELS = {
    'RSA_r_ret':          'Retrieval RSA  (ret_i × ret_j)',
    'RSA_r_enc':          'Encoding RSA   (enc_i × enc_j)',
    'RSA_r_enc_i_ret_j':  'Cross RSA  (enc_i × ret_j)',
    'RSA_r_enc_j_ret_i':  'Cross RSA  (enc_j × ret_i)',
}

# Full predictor set.  Spatial experiments use both; FR1 uses T_lag only.
ALL_PREDICTORS    = ['SP_lag', 'T_lag']
SPATIAL_PREDICTORS = ['SP_lag', 'T_lag']
NONSPATIAL_PREDICTORS = ['T_lag']

BANDS = ['theta', 'alpha', 'beta', 'gamma']

# Cross-covariate used in Model 2 when BOTH predictors are available.
# When only T_lag is available (no spatial task), Model 2 has no cross-covariate —
# it just controls for output_lag [+ semantic_sim].
CROSS_COVARIATE = {
    'T_lag':  'SP_lag',
    'SP_lag': 'T_lag',
}

PRED_LABELS = {
    'T_lag':  'Temporal Lag (T_lag)',
    'SP_lag': 'Spatial Lag (SP_lag)',
}
MODEL_LABELS = {
    'Model1': 'Bare',
    'Model2': 'Controlled',
}

# ---- Palette ----------------------------------------------------------------
BG_COLOR    = 'white'
AX_COLOR    = '#F7F7F7'
GRID_COLOR  = '#DDDDDD'
TEXT_COLOR  = '#222222'
SPINE_COLOR = '#AAAAAA'

REGION_COLORS = {
    'LHP':  '#1A3A6B',
    'RHP':  '#8B1A1A',
    'LLTC': '#1A6B3A',
    'RLTC': '#6B3A1A',
    'LMTL': '#3A1A6B',
    'RMTL': '#6B1A5A',
}
BAND_COLORS = {
    'theta': '#4575B4',
    'alpha': '#74ADD1',
    'beta':  '#F46D43',
    'gamma': '#D73027',
}

# ============================================================================
# LMM FITTING
# ============================================================================

def fit_lmm(df: pd.DataFrame,
            pred_cols: List[str],
            label: str,
            outcome: str,
            formula_rhs: Optional[str] = None) -> Tuple[Optional[object], int]:
    df = df.copy()
    df['subj_sess'] = (df['subject'].astype(str)
                       + '_' + df['session'].astype(str))

    real_cols = [c for c in pred_cols if c in df.columns]
    keep      = [outcome] + real_cols + ['subject', 'subj_sess']
    df        = df[keep].dropna()

    if len(df) < 20:
        print(f"    [{label}] Too few rows ({len(df)}) — skipping")
        return None, 0

    rhs     = formula_rhs if formula_rhs else ' + '.join(real_cols)
    formula = f"{outcome} ~ {rhs}"
    print(f"    [{label}] {formula}  |  N={len(df):,}")

    model = MixedLM.from_formula(
        formula,
        data       = df,
        groups     = df['subject'],
        vc_formula = {'subj_sess': '0 + C(subj_sess)'},
    )

    result = None
    for method in ['lbfgs', 'nm', 'powell']:
        try:
            result = model.fit(reml=True, method=method)
            if np.isfinite(result.llf):
                print(f"    [{label}] optimizer={method}  "
                      f"converged={getattr(result, 'converged', None)}  "
                      f"llf={result.llf:.4f}  AIC={result.aic:.4f}")
                break
            else:
                print(f"    [{label}] llf=NaN with {method}, trying next …")
        except Exception as e:
            print(f"    [{label}] {method} failed: {e}")
            result = None

    if result is None or not np.isfinite(result.llf):
        print(f"    [{label}] WARNING: fit unsuccessful.")
    return result, len(df)


# ============================================================================
# RESULT EXTRACTION
# ============================================================================

def extract_rows(result,
                 pred_display: Dict[str, str],
                 model_label: str,
                 predictor: str,
                 band: str,
                 region: str,
                 outcome: str,
                 experiment: str) -> pd.DataFrame:
    if result is None:
        return pd.DataFrame()
    rows = []
    for col, name in pred_display.items():
        matched = col if col in result.params.index else None
        if matched is None:
            hits    = [k for k in result.params.index
                       if col.lower() in k.lower()]
            matched = hits[0] if hits else None
        if matched is None:
            print(f"    WARNING: '{col}' not found in params — skipping")
            continue
        rows.append({
            'experiment':            experiment,
            'outcome':               outcome,
            'region':                region,
            'predictor_of_interest': predictor,
            'band':                  band,
            'model':                 model_label,
            'term':                  name,
            'col':                   col,
            'beta':                  result.params[matched],
            'se':                    result.bse[matched],
            'z':                     result.tvalues[matched],
            'p_raw':                 result.pvalues[matched],
            'llf':                   result.llf,
            'aic':                   result.aic,
            'nobs':                  int(result.nobs),
        })
    return pd.DataFrame(rows)


def apply_fdr_within_model(df: pd.DataFrame) -> pd.DataFrame:
    if df.empty:
        return df
    _, df['p_fdr'] = fdrcorrection(df['p_raw'].values)
    return df


# ============================================================================
# TEXT FORMATTING
# ============================================================================

def sig_stars(p: float) -> str:
    return ('***' if p < 0.001 else
            '**'  if p < 0.01  else
            '*'   if p < 0.05  else
            '†'   if p < 0.10  else '')


def format_block(title: str, rows_df: pd.DataFrame,
                 outcome: str, experiment: str) -> str:
    sep  = '=' * 92
    sep2 = '-' * 92
    hdr  = (f"{'Term':<40} {'β':>8} {'SE':>8} {'z':>8} "
            f"{'p_raw':>10} {'p_fdr':>10} {'AIC':>10} {'N':>8} {'sig':>5}")
    lines = [sep, title, sep2, hdr, sep2]
    for _, row in rows_df.iterrows():
        aic_s = (f"{row['aic']:>10.2f}"
                 if pd.notna(row.get('aic')) else '       NaN')
        p_fdr = row.get('p_fdr', np.nan)
        lines.append(
            f"{row['term']:<40} {row['beta']:>8.4f} {row['se']:>8.4f} "
            f"{row['z']:>8.3f} {row['p_raw']:>10.4f} "
            f"{p_fdr:>10.4f} {aic_s} {int(row['nobs']):>8,} "
            f"{sig_stars(p_fdr):>5}"
        )
    lines += [sep2,
              'FDR: BH correction within each model across terms.',
              '† p<.10  * p<.05  ** p<.01  *** p<.001',
              f'Outcome = {outcome}.  Experiment = {experiment}.',
              'All continuous predictors on raw scale.',
              sep]
    return '\n'.join(lines)


# ============================================================================
# PLOTTING HELPERS
# ============================================================================

def _style_ax(ax):
    ax.set_facecolor(AX_COLOR)
    ax.tick_params(colors=TEXT_COLOR, labelsize=8.5)
    ax.xaxis.label.set_color(TEXT_COLOR)
    ax.yaxis.label.set_color(TEXT_COLOR)
    ax.title.set_color(TEXT_COLOR)
    ax.grid(True, color=GRID_COLOR, lw=0.6, zorder=0)
    for spine in ax.spines.values():
        spine.set_edgecolor(SPINE_COLOR)
        spine.set_linewidth(0.8)


def plot_forest(all_results: pd.DataFrame,
                region: str,
                predictor: str,
                band: str,
                outcome: str,
                experiment: str,
                save_path: Path):
    df = all_results[
        (all_results['region']                == region) &
        (all_results['predictor_of_interest'] == predictor) &
        (all_results['band']                  == band) &
        (all_results['outcome']               == outcome) &
        (all_results['col']                   == predictor)
    ].copy()

    if df.empty:
        return

    df['ci_lo'] = df['beta'] - 1.96 * df['se']
    df['ci_hi'] = df['beta'] + 1.96 * df['se']

    models = [m for m in ['Model1', 'Model2'] if m in df['model'].values]
    color  = REGION_COLORS.get(region, '#444444')

    fig, ax = plt.subplots(figsize=(9, max(3, len(models) * 1.4 + 1)))
    fig.patch.set_facecolor(BG_COLOR)
    _style_ax(ax)

    y_pos, yticks, ylabels = 0, [], []
    for model in models:
        row = df[df['model'] == model]
        if row.empty:
            continue
        row  = row.iloc[0]
        xerr = [[row['beta'] - row['ci_lo']], [row['ci_hi'] - row['beta']]]
        ax.errorbar(row['beta'], y_pos, xerr=xerr,
                    fmt='o', color=color, ecolor=color,
                    elinewidth=1.5, capsize=4, capthick=1.5,
                    markersize=7, zorder=3)
        p_show = row.get('p_fdr', row['p_raw'])
        stars  = sig_stars(p_show)
        if stars:
            offset = abs(row['ci_hi'] - row['beta']) * 0.15 + 0.001
            ax.text(row['ci_hi'] + offset, y_pos, stars,
                    color=color, va='center', fontsize=9, fontweight='bold')
        yticks.append(y_pos)
        ylabels.append(MODEL_LABELS.get(model, model))
        y_pos -= 1

    ax.axvline(0, color=SPINE_COLOR, lw=1.2, ls='--', zorder=1)
    ax.set_yticks(yticks)
    ax.set_yticklabels(ylabels, fontsize=9)
    ax.set_xlabel(f'β  ({PRED_LABELS[predictor]})', fontsize=10)
    ax.set_title(
        f"{region}  |  {OUTCOME_LABELS[outcome]} ~ {PRED_LABELS[predictor]}  |  "
        f"{band.capitalize()} band  [{experiment}]",
        fontsize=10, fontweight='bold', pad=8)

    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"    ✓ Forest → {save_path.name}")


def _subject_slopes(hd: pd.DataFrame,
                    predictor: str,
                    outcome: str) -> pd.DataFrame:
    rows = []
    for subj, sg in hd.groupby('subject'):
        sg = sg.dropna(subset=[predictor, outcome])
        if len(sg) < 3:
            continue
        x = sg[predictor].values.astype(float)
        y = sg[outcome].values.astype(float)
        if x.std() == 0:
            continue
        m, b = np.polyfit(x, y, 1)
        rows.append({'subject': subj, 'slope': m,
                     'intercept': b, 'n_pairs': len(sg)})
    return pd.DataFrame(rows)


def _subject_zscore_rsa(hd: pd.DataFrame, outcome: str) -> pd.DataFrame:
    hd = hd.copy()
    def zscore(x):
        s = x.std()
        return (x - x.mean()) / s if s > 0 else x - x.mean()
    hd[f'{outcome}_z'] = hd.groupby('subject')[outcome].transform(zscore)
    return hd


def plot_interaction(region: str,
                     predictor: str,
                     band: str,
                     outcome: str,
                     raw_df: pd.DataFrame,
                     experiment: str,
                     save_path: Path):
    outcome_z = f'{outcome}_z'

    sub_df = raw_df[
        raw_df['band'] == band
    ].copy().dropna(subset=[predictor, outcome])

    if sub_df.empty:
        return

    color       = REGION_COLORS.get(region, '#444444')
    is_discrete = (predictor == 'T_lag')

    fig, (ax_sp, ax_sl) = plt.subplots(2, 1, figsize=(10, 10))
    fig.patch.set_facecolor(BG_COLOR)
    _style_ax(ax_sp)
    _style_ax(ax_sl)

    n_subj  = sub_df['subject'].nunique()
    n_pairs = len(sub_df)

    x_all = sub_df[predictor].values.astype(float)
    y_all = sub_df[outcome].values.astype(float)
    r_val, p_val = pearsonr(x_all, y_all)
    p_str = f'p={p_val:.3f}' if p_val >= 0.001 else 'p<0.001'

    hd = _subject_zscore_rsa(sub_df, outcome)

    if is_discrete:
        MAX_VALS = 20
        top_vals = sorted(
            hd[predictor].value_counts().nlargest(MAX_VALS).index.tolist())
        x_grid = np.array(top_vals, dtype=float)
    else:
        N_BINS     = 12
        hd['_bin'] = pd.cut(hd[predictor], bins=N_BINS)
        bin_mids   = (hd.groupby('_bin', observed=True)[predictor]
                       .mean().dropna())
        x_grid     = bin_mids.values

    subj_lines = []
    for subj, sg in hd.groupby('subject'):
        sg = sg.dropna(subset=[predictor, outcome_z])
        if len(sg) < 3:
            continue
        if is_discrete:
            pts = (sg.groupby(predictor)[outcome_z]
                     .mean().reindex(top_vals))
        else:
            pts = (sg.groupby('_bin', observed=True)[outcome_z].mean())
            pts.index = pts.index.map(
                lambda b: b.mid if hasattr(b, 'mid') else np.nan)
            pts = pts.dropna()
        pts = pts.dropna()
        if len(pts) < 2:
            continue
        subj_lines.append(pts.values)
        ax_sp.plot(x_grid[:len(pts.values)], pts.values,
                   color=color, alpha=0.15, lw=0.9, zorder=2)

    if subj_lines:
        max_len  = max(len(s) for s in subj_lines)
        padded   = np.array([
            np.pad(s.astype(float), (0, max_len - len(s)),
                   constant_values=np.nan)
            for s in subj_lines
        ])
        grp_mean = np.nanmean(padded, axis=0)
        grp_sem  = (np.nanstd(padded, axis=0)
                    / np.sqrt((~np.isnan(padded)).sum(axis=0)))
        xg = x_grid[:max_len]

        ax_sp.fill_between(xg, grp_mean - grp_sem, grp_mean + grp_sem,
                           color=color, alpha=0.20, zorder=3)
        ax_sp.plot(xg, grp_mean, color=color, lw=2.5, zorder=4,
                   label=f'Group mean ± SEM  (n={n_subj} subj)')

        valid = np.isfinite(grp_mean) & np.isfinite(xg)
        if valid.sum() >= 2:
            m_z, b_z = np.polyfit(xg[valid], grp_mean[valid], 1)
            ax_sp.plot(xg[valid], m_z * xg[valid] + b_z,
                       color=TEXT_COLOR, lw=1.5, ls='--', alpha=0.6,
                       zorder=5, label=f'OLS  r={r_val:.3f} {p_str}')

    ax_sp.axhline(0, color=SPINE_COLOR, lw=1, ls=':', zorder=1)
    if is_discrete:
        ax_sp.set_xticks(x_grid)
        ax_sp.set_xticklabels(
            [str(int(v)) for v in x_grid], fontsize=7,
            rotation=45 if len(x_grid) > 12 else 0)
    ax_sp.set_xlabel(PRED_LABELS[predictor], fontsize=9)
    ax_sp.set_ylabel(f'{outcome}  (z-scored within subject)', fontsize=9)
    ax_sp.set_title(
        f'{region}  |  {band.capitalize()} band  |  {experiment}\n'
        f'Outcome: {OUTCOME_LABELS[outcome]}  |  '
        f'n={n_subj} subj, {n_pairs:,} pairs',
        fontsize=10, fontweight='bold')
    ax_sp.legend(facecolor='white', edgecolor=SPINE_COLOR, fontsize=8)

    slopes_df = _subject_slopes(hd, predictor, outcome)

    if slopes_df.empty:
        ax_sl.set_visible(False)
    else:
        slopes     = slopes_df['slope'].values
        n_sl       = len(slopes)
        mean_slope = slopes.mean()
        sem_slope  = slopes.std() / np.sqrt(n_sl)
        ci95       = 1.96 * sem_slope
        n_neg      = (slopes < 0).sum()
        n_pos      = (slopes > 0).sum()

        order      = np.argsort(slopes)
        y_pos_sl   = np.arange(n_sl)
        dot_colors = [REGION_COLORS.get(region, '#444444')
                      if s >= 0 else '#888888'
                      for s in slopes[order]]

        ax_sl.scatter(slopes[order], y_pos_sl,
                      c=dot_colors, s=40, zorder=3,
                      edgecolors='white', linewidths=0.4)
        for yi, si, dc in zip(y_pos_sl, slopes[order], dot_colors):
            ax_sl.plot([0, si], [yi, yi],
                       color=dc, alpha=0.35, lw=0.8, zorder=2)

        ax_sl.axvline(0, color=SPINE_COLOR, lw=1.2, ls='--', zorder=1)
        ax_sl.axvspan(mean_slope - ci95, mean_slope + ci95,
                      color=color, alpha=0.12, zorder=0)
        ax_sl.axvline(mean_slope, color=color, lw=2.5, zorder=4,
                      label=f'Mean β={mean_slope:.4f} ± {ci95:.4f} (95%CI)')

        t_stat, p_1samp = ttest_1samp(slopes, 0)
        p1_str = f'p={p_1samp:.3f}' if p_1samp >= 0.001 else 'p<0.001'
        stars  = sig_stars(p_1samp)
        ax_sl.text(mean_slope, n_sl + 0.3,
                   f'{stars}  {p1_str}' if stars else p1_str,
                   ha='center', va='bottom',
                   color=color, fontsize=9, fontweight='bold')

        ax_sl.set_yticks([])
        ax_sl.set_xlabel(
            f'Subject-level OLS slope  ({outcome} ~ {predictor})', fontsize=9)
        ax_sl.set_title(
            f'{region}  |  {band.capitalize()} band  |  {OUTCOME_LABELS[outcome]}\n'
            f'Each dot = 1 subject  '
            f'({n_neg}/{n_sl} negative, {n_pos}/{n_sl} positive)',
            fontsize=10, fontweight='bold')
        ax_sl.legend(facecolor='white', edgecolor=SPINE_COLOR, fontsize=8)
        pct_neg = n_neg / n_sl * 100
        ax_sl.text(0.02, 0.04,
                   f'{pct_neg:.0f}% of subjects show negative slope',
                   transform=ax_sl.transAxes,
                   fontsize=8, color=TEXT_COLOR, style='italic')

    fig.suptitle(
        f"{outcome} ~ {PRED_LABELS[predictor]}  |  {region}  |  "
        f"{band.capitalize()} band\n"
        f"Top: z-scored spaghetti  |  Bottom: per-subject slopes",
        fontsize=12, fontweight='bold', y=1.01, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"    ✓ Interaction → {save_path.name}")


def plot_summary_heatmap(all_results: pd.DataFrame,
                         region: str,
                         outcome: str,
                         experiment: str,
                         predictors: List[str],
                         save_path: Path):
    """β heatmap: rows = predictors, cols = bands, Model 1 only."""
    pivot_rows = []
    for pred in predictors:
        for band in BANDS:
            sub = all_results[
                (all_results['region']                == region) &
                (all_results['predictor_of_interest'] == pred) &
                (all_results['band']                  == band) &
                (all_results['outcome']               == outcome) &
                (all_results['model']                 == 'Model1') &
                (all_results['col']                   == pred)
            ]
            if sub.empty:
                beta, stars = np.nan, ''
            else:
                r     = sub.iloc[0]
                beta  = r['beta']
                stars = sig_stars(r.get('p_fdr', r['p_raw']))
            pivot_rows.append({'predictor': PRED_LABELS[pred],
                                'band': band, 'beta': beta, 'stars': stars})

    piv       = pd.DataFrame(pivot_rows)
    beta_mat  = piv.pivot(index='predictor', columns='band', values='beta')[BANDS]
    stars_mat = piv.pivot(index='predictor', columns='band', values='stars')[BANDS]

    fig, ax = plt.subplots(figsize=(9, max(2.5, len(predictors) * 1.2 + 0.8)))
    fig.patch.set_facecolor(BG_COLOR)
    vmax = np.nanmax(np.abs(beta_mat.values)) or 1.0
    im   = ax.imshow(beta_mat.values.astype(float),
                     aspect='auto', cmap='RdBu_r',
                     vmin=-vmax, vmax=vmax)

    for i in range(beta_mat.shape[0]):
        for j in range(beta_mat.shape[1]):
            val  = beta_mat.values[i, j]
            star = stars_mat.values[i, j]
            if not np.isnan(val):
                cell_norm = (val + vmax) / (2 * vmax)
                txt_color = 'white' if (cell_norm < 0.35 or cell_norm > 0.65) else TEXT_COLOR
                ax.text(j, i, f"{val:.3f}\n{star}",
                        ha='center', va='center',
                        color=txt_color, fontsize=9, fontweight='bold')

    ax.set_xticks(range(len(BANDS)))
    ax.set_xticklabels([b.capitalize() for b in BANDS], fontsize=10)
    ax.set_yticks(range(len(beta_mat.index)))
    ax.set_yticklabels(beta_mat.index.tolist(), fontsize=9)
    ax.tick_params(colors=TEXT_COLOR)
    for spine in ax.spines.values():
        spine.set_edgecolor(SPINE_COLOR)

    cbar = fig.colorbar(im, ax=ax, fraction=0.04, pad=0.02)
    cbar.set_label(f'β  (predictor → {outcome})', fontsize=9, color=TEXT_COLOR)
    cbar.ax.yaxis.set_tick_params(color=TEXT_COLOR)
    plt.setp(cbar.ax.yaxis.get_ticklabels(), color=TEXT_COLOR)

    ax.set_title(
        f'{region}  |  {OUTCOME_LABELS[outcome]} ~ predictor  |  '
        f'β (Model 1)  [{experiment}]',
        fontsize=10, fontweight='bold', pad=8, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"  ✓ Heatmap → {save_path.name}")


def plot_beta_bars(all_results: pd.DataFrame,
                   region: str,
                   outcome: str,
                   experiment: str,
                   predictors: List[str],
                   save_path: Path):
    """Bar chart of β by band, one column per predictor, two rows (Model1/2)."""
    n_cols = len(predictors)
    fig, axes = plt.subplots(2, n_cols, figsize=(7 * n_cols, 8))
    fig.patch.set_facecolor(BG_COLOR)
    # Normalise axes to always be 2-D
    if n_cols == 1:
        axes = axes[:, np.newaxis]

    x     = np.arange(len(BANDS))

    for col_idx, pred in enumerate(predictors):
        for row_idx, model_key in enumerate(['Model1', 'Model2']):
            ax = axes[row_idx, col_idx]
            _style_ax(ax)

            betas, errors, pvals = [], [], []
            for band in BANDS:
                sub = all_results[
                    (all_results['region']                == region) &
                    (all_results['predictor_of_interest'] == pred) &
                    (all_results['outcome']               == outcome) &
                    (all_results['model']                 == model_key) &
                    (all_results['band']                  == band) &
                    (all_results['col']                   == pred)
                ]
                if sub.empty:
                    betas.append(np.nan); errors.append(0); pvals.append(1.0)
                else:
                    r = sub.iloc[0]
                    betas.append(r['beta'])
                    errors.append(r['se'])
                    pvals.append(r.get('p_fdr', r['p_raw']))

            plot_betas  = [b if np.isfinite(b) else 0 for b in betas]
            plot_errors = [e if np.isfinite(betas[i]) else 0
                           for i, e in enumerate(errors)]

            bars = ax.bar(x, plot_betas, 0.50,
                          color=[BAND_COLORS[b] for b in BANDS],
                          yerr=plot_errors,
                          error_kw=dict(ecolor=TEXT_COLOR, capsize=3, elinewidth=1),
                          alpha=0.82)

            for bar, beta, p in zip(bars, betas, pvals):
                if not np.isfinite(beta):
                    continue
                stars = sig_stars(p)
                if stars:
                    h    = bar.get_height()
                    sign = 1 if h >= 0 else -1
                    ax.text(bar.get_x() + bar.get_width() / 2,
                            h + sign * max(abs(h) * 0.05, 0.005),
                            stars, ha='center', va='bottom',
                            color=TEXT_COLOR, fontsize=9, fontweight='bold')

            ax.axhline(0, color=SPINE_COLOR, lw=1.0, ls='--')
            ax.set_xticks(x)
            ax.set_xticklabels([b.capitalize() for b in BANDS], fontsize=9)
            ax.set_xlabel('Frequency Band', fontsize=9)
            ax.set_ylabel('β', fontsize=9)
            ax.set_title(
                f"{region}  |  {outcome} ~ {PRED_LABELS[pred]}\n"
                f"[{MODEL_LABELS[model_key]}]",
                fontsize=9, fontweight='bold')

    pred_labels_str = ' / '.join(PRED_LABELS[p] for p in predictors)
    fig.suptitle(
        f'{region}  |  {OUTCOME_LABELS[outcome]} ~ {pred_labels_str}  |  '
        f'β by Frequency Band  [{experiment}]',
        fontsize=12, fontweight='bold', y=1.01, color=TEXT_COLOR)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"  ✓ Beta bars → {save_path.name}")


def plot_region_comparison(all_results: pd.DataFrame,
                            outcome: str,
                            predictor: str,
                            band: str,
                            experiment: str,
                            save_path: Path):
    sub = all_results[
        (all_results['outcome']               == outcome) &
        (all_results['predictor_of_interest'] == predictor) &
        (all_results['band']                  == band) &
        (all_results['col']                   == predictor)
    ].copy()

    if sub.empty:
        return

    sub['ci_lo'] = sub['beta'] - 1.96 * sub['se']
    sub['ci_hi'] = sub['beta'] + 1.96 * sub['se']

    regions_present = [r for r in REGIONS if r in sub['region'].values]
    if not regions_present:
        return

    fig, ax = plt.subplots(
        figsize=(10, max(4, len(regions_present) * 1.6 + 1.5)))
    fig.patch.set_facecolor(BG_COLOR)
    _style_ax(ax)

    y_pos   = 0
    yticks  = []
    ylabels = []
    offsets = {'Model1': -0.18, 'Model2': 0.18}
    markers = {'Model1': 'o', 'Model2': 's'}

    for region in regions_present:
        color = REGION_COLORS.get(region, '#444444')
        for model_key in ['Model1', 'Model2']:
            row = sub[(sub['region'] == region) & (sub['model'] == model_key)]
            if row.empty:
                continue
            row  = row.iloc[0]
            yp   = y_pos + offsets[model_key]
            xerr = [[row['beta'] - row['ci_lo']], [row['ci_hi'] - row['beta']]]
            ax.errorbar(row['beta'], yp, xerr=xerr,
                        fmt=markers[model_key], color=color, ecolor=color,
                        elinewidth=1.4, capsize=3, capthick=1.4,
                        markersize=6, zorder=3)
            p_show = row.get('p_fdr', row['p_raw'])
            stars  = sig_stars(p_show)
            if stars:
                offset_x = abs(row['ci_hi'] - row['beta']) * 0.15 + 0.001
                ax.text(row['ci_hi'] + offset_x, yp, stars,
                        color=color, va='center', fontsize=8, fontweight='bold')

        yticks.append(y_pos)
        ylabels.append(region)
        y_pos -= 1

    ax.axvline(0, color=SPINE_COLOR, lw=1.2, ls='--', zorder=1)
    ax.set_yticks(yticks)
    ax.set_yticklabels(ylabels, fontsize=10)
    ax.set_xlabel(f'β  ({PRED_LABELS[predictor]})', fontsize=10)
    ax.set_title(
        f"All regions  |  {OUTCOME_LABELS[outcome]} ~ {PRED_LABELS[predictor]}\n"
        f"{band.capitalize()} band  |  ○ = Bare  □ = Controlled  [{experiment}]",
        fontsize=10, fontweight='bold', pad=8)
    fig.tight_layout()
    fig.savefig(save_path, dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.close(fig)
    print(f"  ✓ Region comparison → {save_path.name}")


# ============================================================================
# DATA LOADING
# ============================================================================

def load_data(experiment: str) -> Optional[pd.DataFrame]:
    fpath = INPUT_DIR / f"ALL_SUBJECTS_{experiment}_allregions_allbands_rsa_lag.csv"

    if not fpath.exists():
        print(f"  ✗ Master CSV not found: {fpath}")
        return None

    print(f"  Loading {fpath.name} …")
    df = pd.read_csv(fpath)
    print(f"  Loaded {len(df):,} rows")

    missing_outcomes = [o for o in OUTCOMES if o not in df.columns]
    if missing_outcomes:
        print(f"  ✗ Missing outcome columns: {missing_outcomes}")
        return None

    df = df[df['region'].isin(REGIONS)].copy()
    print(f"  After region filter {REGIONS}: {len(df):,} rows")

    if df.empty:
        print("  ✗ No data after filtering.")
        return None

    df['subject'] = experiment + '_' + df['subject'].astype(str)

    print(f"\n  Rows         : {len(df):,}")
    print(f"  Subjects     : {df['subject'].nunique()}")
    print(f"  Regions      : {sorted(df['region'].unique().tolist())}")
    print(f"  Bands        : {sorted(df['band'].unique().tolist())}")
    print(f"  Outcomes     : {[o for o in OUTCOMES if o in df.columns]}")
    return df


# ============================================================================
# PER-COMBINATION ANALYSIS
# ============================================================================

def run_analysis_for_combination(df: pd.DataFrame,
                                  region: str,
                                  predictor: str,
                                  band: str,
                                  outcome: str,
                                  has_semsim: bool,
                                  has_spatial: bool,
                                  experiment: str,
                                  ) -> Tuple[pd.DataFrame, str]:
    """
    Fit Model1 and Model2 for one (region, predictor, band, outcome) cell.

    Model 2 cross-covariate logic
    ─────────────────────────────
    Spatial experiment  : outcome ~ predictor + cross_cov + output_lag [+ sem]
    Non-spatial (FR1)   : outcome ~ T_lag + output_lag [+ sem]
                          (no SP_lag to partial out; cross_cov simply dropped)
    """
    sub = df[
        (df['region'] == region) &
        (df['band']   == band)
    ].copy()

    if sub.empty:
        return pd.DataFrame(), ""

    print(f"\n  ── {region}  |  {outcome} ~ {predictor}  |  {band.capitalize()} ──")
    print(f"     Rows: {len(sub):,}  |  Subjects: {sub['subject'].nunique()}")

    all_rows    = []
    text_blocks = [
        f"EXPERIMENT: {experiment}  OUTCOME: {outcome}  "
        f"REGION: {region}  PREDICTOR: {predictor}  BAND: {band}",
        f"N rows: {len(sub):,}  |  subjects: {sub['subject'].nunique()}\n",
    ]

    def run_and_collect(real_cols, model_key, title, pred_display,
                        formula_rhs=None):
        res, _ = fit_lmm(sub, real_cols, model_key, outcome,
                         formula_rhs=formula_rhs)
        rows   = extract_rows(res, pred_display, model_key,
                              predictor, band, region, outcome, experiment)
        rows   = apply_fdr_within_model(rows)
        if not rows.empty:
            all_rows.append(rows)
            block = format_block(title, rows, outcome, experiment)
            text_blocks.append(block)
            print('\n' + block)
        return res

    # ── Model 1: bare ────────────────────────────────────────────────────────
    run_and_collect(
        [predictor],
        'Model1',
        f'Model 1 — {outcome} ~ {predictor}  [{region} / {band}]',
        {predictor: PRED_LABELS[predictor]},
    )

    # ── Model 2: controlled ───────────────────────────────────────────────────
    if has_spatial:
        # Standard: partial out cross-covariate + output_lag [+ semsim]
        cross_cov   = CROSS_COVARIATE[predictor]
        ctrl_cols   = [predictor, cross_cov, 'output_lag']
        ctrl_disp   = {
            predictor:    PRED_LABELS[predictor],
            cross_cov:    f'{PRED_LABELS[cross_cov]}  [covariate]',
            'output_lag': 'output_lag  [covariate]',
        }
        m2_title = (f'Model 2 — {outcome} ~ {predictor} + {cross_cov} + controls'
                    f'  [{region} / {band}]')
    else:
        # No spatial task: drop SP_lag cross-covariate entirely
        ctrl_cols   = [predictor, 'output_lag']
        ctrl_disp   = {
            predictor:    PRED_LABELS[predictor],
            'output_lag': 'output_lag  [covariate]',
        }
        m2_title = (f'Model 2 — {outcome} ~ {predictor} + output_lag + controls'
                    f'  [{region} / {band}]  [no SP_lag — {experiment}]')

    if has_semsim:
        ctrl_cols.append('semantic_sim')
        ctrl_disp['semantic_sim'] = 'semantic_sim  [covariate]'

    run_and_collect(ctrl_cols, 'Model2', m2_title, ctrl_disp)

    result_df = (pd.concat(all_rows, ignore_index=True)
                 if all_rows else pd.DataFrame())
    return result_df, '\n\n'.join(text_blocks)


# ============================================================================
# PER-EXPERIMENT RUNNER
# ============================================================================

def run_experiment(experiment: str):
    cfg         = EXPERIMENT_CONFIG.get(experiment)
    if cfg is None:
        print(f"\n✗ Unknown experiment '{experiment}'. "
              f"Add it to EXPERIMENT_CONFIG.")
        return
    has_spatial = cfg['has_spatial']

    # Active predictors for this experiment
    predictors = SPATIAL_PREDICTORS if has_spatial else NONSPATIAL_PREDICTORS

    # Per-experiment output dirs
    output_dir = BASE_OUTPUT_DIR / experiment
    plot_dir   = output_dir / 'plots'
    plot_dir.mkdir(parents=True, exist_ok=True)

    print(f"\n{'='*80}")
    print(f"EXPERIMENT : {experiment}")
    print(f"Spatial    : {has_spatial}")
    print(f"Predictors : {predictors}")
    print(f"Outcomes   : {OUTCOMES}")
    print(f"Bands      : {BANDS}")
    print(f"Regions    : {REGIONS}")
    print(f"Output     : {output_dir}")
    print(f"{'='*80}")

    df = load_data(experiment)
    if df is None or df.empty:
        print("No data loaded. Skipping experiment.")
        return

    has_semsim = ('semantic_sim' in df.columns
                  and df['semantic_sim'].notna().any())
    print(f"\n  semantic_sim available : {has_semsim}")
    if not has_spatial:
        print(f"  SP_lag predictor       : SKIPPED (no spatial task)")

    ALL_RESULTS = []

    for outcome in OUTCOMES:
        print(f"\n{'━'*80}")
        print(f"OUTCOME: {outcome}  ({OUTCOME_LABELS[outcome]})")
        print(f"{'━'*80}")

        outcome_plot_dir = plot_dir / outcome
        outcome_plot_dir.mkdir(exist_ok=True)

        for region in REGIONS:
            region_results = []
            print(f"\n{'─'*80}")
            print(f"REGION: {region}  |  OUTCOME: {outcome}")
            print(f"{'─'*80}")

            for predictor in predictors:
                for band in BANDS:
                    res_df, text = run_analysis_for_combination(
                        df, region, predictor, band, outcome,
                        has_semsim, has_spatial, experiment)

                    if not res_df.empty:
                        region_results.append(res_df)
                        ALL_RESULTS.append(res_df)

                    tag      = f"{region}_{predictor}_{band}_{outcome}"
                    csv_path = output_dir / f"LMM_{tag}_results.csv"
                    txt_path = output_dir / f"LMM_{tag}_results.txt"

                    if not res_df.empty:
                        res_df.to_csv(csv_path, index=False)
                    if text:
                        with open(txt_path, 'w') as f:
                            f.write(text)

                    if not res_df.empty:
                        plot_forest(
                            res_df, region, predictor, band, outcome,
                            experiment,
                            outcome_plot_dir / f"forest_{tag}.png")

                    region_band_df = df[
                        (df['region'] == region) & (df['band'] == band)
                    ].copy()
                    if not region_band_df.empty:
                        plot_interaction(
                            region, predictor, band, outcome, region_band_df,
                            experiment,
                            outcome_plot_dir / f"interaction_{tag}.png")

            if region_results:
                reg_df = pd.concat(region_results, ignore_index=True)
                plot_summary_heatmap(
                    reg_df, region, outcome, experiment, predictors,
                    outcome_plot_dir / f"summary_heatmap_{region}_{outcome}.png")
                plot_beta_bars(
                    reg_df, region, outcome, experiment, predictors,
                    outcome_plot_dir / f"beta_bars_{region}_{outcome}.png")

        # Cross-region comparison plots — only for predictors active this experiment
        if ALL_RESULTS:
            so_far         = pd.concat(ALL_RESULTS, ignore_index=True)
            so_far_outcome = so_far[so_far['outcome'] == outcome]
            if not so_far_outcome.empty:
                for predictor in predictors:
                    for band in BANDS:
                        plot_region_comparison(
                            so_far_outcome, outcome, predictor, band,
                            experiment,
                            outcome_plot_dir /
                            f"region_comparison_{outcome}_{predictor}_{band}.png")

    # ── Master CSV ────────────────────────────────────────────────────────────
    if ALL_RESULTS:
        master_df   = pd.concat(ALL_RESULTS, ignore_index=True)
        master_path = output_dir / 'LMM_ALL_results.csv'
        master_df.to_csv(master_path, index=False)

        print(f"\n{'='*80}")
        print(f"✓ Master results → {master_path}")
        print(f"  Rows      : {len(master_df):,}")
        print(f"  Outcomes  : {sorted(master_df['outcome'].unique())}")
        print(f"  Regions   : {sorted(master_df['region'].unique())}")
        print(f"  Bands     : {sorted(master_df['band'].unique())}")
        print(f"  Predictors: {sorted(master_df['predictor_of_interest'].unique())}")

    print(f"\n{'='*80}")
    print(f"✓ ANALYSIS COMPLETE  [{experiment}]")
    print(f"  Results : {output_dir}")
    print(f"  Plots   : {plot_dir}")
    print(f"{'='*80}")


# ============================================================================
# MAIN
# ============================================================================

if __name__ == '__main__':
    # Allow a single experiment to be passed on the command line, e.g.:
    #   python rsa_lag_lmm_allregions_fr1.py FR1
    # Ignore Jupyter/IPython kernel flags (e.g. -f kernel-xxx.json)
    known_exps = [a for a in sys.argv[1:]
                  if a in EXPERIMENT_CONFIG]
    if known_exps:
        exps = known_exps
    else:
        exps = EXPERIMENTS_TO_RUN

    for exp in exps:
        run_experiment(exp)

    print(f"\n{'='*80}")
    print("✓ ALL EXPERIMENTS COMPLETE")
    print(f"{'='*80}")